# Uncertainty Sampling in Active Learning - Starter Notebook
This notebook compares least-confidence, margin, and entropy uncertainty criteria for selecting query points.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
X, y = make_classification(n_samples=900, n_features=8, n_informative=6, random_state=42)
X_pool, _, y_pool, _ = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_pool[:80], y_pool[:80])
probs = model.predict_proba(X_pool[80:])

least_confidence = 1 - probs.max(axis=1)
margin = np.abs(probs[:, 0] - probs[:, 1])
entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)

df = pd.DataFrame({
    'least_confidence': least_confidence,
    'margin': margin,
    'entropy': entropy,
})

summary = pd.DataFrame({
    'criterion': ['least_confidence', 'small_margin', 'highest_entropy'],
    'selected_indices': [
        df.nlargest(10, 'least_confidence').index.tolist(),
        df.nsmallest(10, 'margin').index.tolist(),
        df.nlargest(10, 'entropy').index.tolist(),
    ],
})
summary